In [1]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from collections import defaultdict, Counter



BASE_DIR = "/home/aninotna/magister/tesis/justh2_pipeline"
DATA_DIR = os.path.join(BASE_DIR, "data")
PLOTS_DIR = os.path.join(BASE_DIR, "plots", "cluster_characterization")
os.makedirs(PLOTS_DIR, exist_ok=True)

print("Entorno configurado")
print(f"  BASE_DIR: {BASE_DIR}")
print(f"  DATA_DIR: {DATA_DIR}")
print(f"  PLOTS_DIR: {PLOTS_DIR}")

Entorno configurado
  BASE_DIR: /home/aninotna/magister/tesis/justh2_pipeline
  DATA_DIR: /home/aninotna/magister/tesis/justh2_pipeline/data
  PLOTS_DIR: /home/aninotna/magister/tesis/justh2_pipeline/plots/cluster_characterization


## Importación de Resultados de Clustering

Cargamos los resultados exportados desde el notebook `07_experiments_1_clustering.ipynb`.

In [29]:
CLUSTERING_DIR = os.path.join(DATA_DIR, "autoencoder_results", "clustering")

clustering_files = sorted(glob(os.path.join(CLUSTERING_DIR, "clustering_results_*.pkl")))

if not clustering_files:
    raise FileNotFoundError(f"No se encontraron archivos de clustering en {CLUSTERING_DIR}")

latest_file = clustering_files[-1]
print(f"Archivo de clustering más reciente: {os.path.basename(latest_file)}")

with open(latest_file, "rb") as f:
    clustering_data = pickle.load(f)

K_CLUSTERS = clustering_data["K_CLUSTERS"]
SEED = clustering_data["SEED"]
N_PER_SCENARIO = clustering_data["N_PER_SCENARIO"]
MODEL_ORDER = clustering_data["MODEL_ORDER"]
feature_names = clustering_data["feature_names"]

lat = clustering_data["coords"]["lat"]
lon = clustering_data["coords"]["lon"]
extent = clustering_data["coords"]["extent"]

CLUSTERING_RESULTS = clustering_data["CLUSTERING_RESULTS"]
LATENTS = clustering_data["LATENTS"]

X_BASE = clustering_data["data_blocks"]["X_BASE"]
X245_orig = clustering_data["data_blocks"]["X245_orig"]
X370_orig = clustering_data["data_blocks"]["X370_orig"]
X585_orig = clustering_data["data_blocks"]["X585_orig"]

print(f"\nDatos cargados exitosamente:")
print(f"  Timestamp: {clustering_data['timestamp']}")
print(f"  K_CLUSTERS: {K_CLUSTERS}")
print(f"  Modelos: {MODEL_ORDER}")
print(f"  Píxeles por escenario: {N_PER_SCENARIO}")
print(f"  Features climáticas: {len(feature_names)}")
print(f"  Coordenadas: lat={len(lat)}, lon={len(lon)}")

for model_key in MODEL_ORDER:
    results = CLUSTERING_RESULTS[model_key]
    print(f"\n{model_key}:")
    print(f"  Labels BASE: {results['labels_base'].shape}")
    print(f"  Centroides: {results['centroids'].shape}")
    print(f"  Inertia: {results['inertia']:.2f}")

Archivo de clustering más reciente: clustering_results_20251115_064616.pkl

Datos cargados exitosamente:
  Timestamp: 20251115_064616
  K_CLUSTERS: 10
  Modelos: ['AE', 'VAE']
  Píxeles por escenario: 661
  Features climáticas: 47
  Coordenadas: lat=24, lon=42

AE:
  Labels BASE: (1983,)
  Centroides: (10, 8)
  Inertia: 5664.12

VAE:
  Labels BASE: (1983,)
  Centroides: (10, 8)
  Inertia: 1941.74


In [30]:
import pickle
from sklearn.preprocessing import StandardScaler

def fit_base_scaler(Z_base):
    """
    Ajusta un StandardScaler con el espacio latente BASE.
    
    Parámetros
    ----------
    Z_base : np.ndarray
        Espacio latente de BASE (N_base, latent_dim)
    
    Retorna
    -------
    scaler : StandardScaler
        Escalador ajustado con BASE
    """
    scaler = StandardScaler()
    scaler.fit(Z_base)
    
    print(f"Escalador ajustado con BASE")
    print(f"  Shape: {Z_base.shape}")
    print(f"  Media ajustada: {scaler.mean_[:5]}")
    print(f"  Std ajustada: {scaler.scale_[:5]}")
    
    return scaler


def transform_latents_with_base_scaler(Z_base, Z_T245, Z_T370, Z_T585, scaler=None):
    """
    Escala todos los espacios latentes usando el escalador de BASE.
    
    Parámetros
    ----------
    Z_base : np.ndarray
        Espacio latente BASE (N_base, latent_dim)
    Z_T245 : np.ndarray
        Espacio latente SSP245 (N_245, latent_dim)
    Z_T370 : np.ndarray
        Espacio latente SSP370 (N_370, latent_dim)
    Z_T585 : np.ndarray
        Espacio latente SSP585 (N_585, latent_dim)
    scaler : StandardScaler, opcional
        Si se proporciona, usa este escalador. Si no, ajusta uno nuevo con BASE.
    
    Retorna
    -------
    dict
        Diccionario con:
        - 'Z_base_sc': BASE escalado
        - 'Z_T245_sc': SSP245 escalado
        - 'Z_T370_sc': SSP370 escalado
        - 'Z_T585_sc': SSP585 escalado
        - 'scaler': el escalador usado
        - 'verification': verificación de media/std en BASE escalado
    """
    
    # Ajustar escalador con BASE si no se proporciona
    if scaler is None:
        scaler = fit_base_scaler(Z_base)
    
    # Transformar todos los espacios latentes
    Z_base_sc = scaler.transform(Z_base)
    Z_T245_sc = scaler.transform(Z_T245)
    Z_T370_sc = scaler.transform(Z_T370)
    Z_T585_sc = scaler.transform(Z_T585)
    
    # Verificación: media ≈ 0 y std ≈ 1 solo en BASE
    base_mean = Z_base_sc.mean(axis=0)
    base_std = Z_base_sc.std(axis=0)
    
    verification = {
        'base_mean_close_to_zero': np.allclose(base_mean, 0, atol=1e-10),
        'base_std_close_to_one': np.allclose(base_std, 1, atol=1e-10),
        'base_mean': base_mean,
        'base_std': base_std,
        'T245_mean': Z_T245_sc.mean(axis=0),
        'T245_std': Z_T245_sc.std(axis=0),
        'T370_mean': Z_T370_sc.mean(axis=0),
        'T370_std': Z_T370_sc.std(axis=0),
        'T585_mean': Z_T585_sc.mean(axis=0),
        'T585_std': Z_T585_sc.std(axis=0),
    }
    
    print("\nTransformación completada:")
    print(f"  BASE:   {Z_base.shape} → {Z_base_sc.shape}")
    print(f"  SSP245: {Z_T245.shape} → {Z_T245_sc.shape}")
    print(f"  SSP370: {Z_T370.shape} → {Z_T370_sc.shape}")
    print(f"  SSP585: {Z_T585.shape} → {Z_T585_sc.shape}")
    
    print(f"\nVerificación BASE escalado:")
    print(f"  Media ≈ 0: {verification['base_mean_close_to_zero']} (max abs = {np.abs(base_mean).max():.2e})")
    print(f"  Std ≈ 1:   {verification['base_std_close_to_one']} (max diff = {np.abs(base_std - 1).max():.2e})")
    
    print(f"\nEstadísticas escenarios futuros (NO deben ser 0/1):")
    print(f"  SSP245 - Media: {verification['T245_mean'][:5]}")
    print(f"  SSP245 - Std:   {verification['T245_std'][:5]}")
    print(f"  SSP370 - Media: {verification['T370_mean'][:5]}")
    print(f"  SSP370 - Std:   {verification['T370_std'][:5]}")
    print(f"  SSP585 - Media: {verification['T585_mean'][:5]}")
    print(f"  SSP585 - Std:   {verification['T585_std'][:5]}")
    
    return {
        'Z_base_sc': Z_base_sc,
        'Z_T245_sc': Z_T245_sc,
        'Z_T370_sc': Z_T370_sc,
        'Z_T585_sc': Z_T585_sc,
        'scaler': scaler,
        'verification': verification
    }


def save_base_scaler(scaler, filepath):
    """
    Guarda el escalador BASE para uso futuro.
    
    Parámetros
    ----------
    scaler : StandardScaler
        Escalador ajustado con BASE
    filepath : str
        Ruta donde guardar el escalador
    """
    with open(filepath, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"Escalador guardado en: {filepath}")


def load_base_scaler(filepath):
    """
    Carga un escalador BASE previamente guardado.
    
    Parámetros
    ----------
    filepath : str
        Ruta del escalador guardado
    
    Retorna
    -------
    scaler : StandardScaler
        Escalador cargado
    """
    with open(filepath, 'rb') as f:
        scaler = pickle.load(f)
    print(f"Escalador cargado desde: {filepath}")
    print(f"  Media: {scaler.mean_[:5]}")
    print(f"  Std: {scaler.scale_[:5]}")
    return scaler


print("Utilidades de escalado consistente cargadas")
print("\nFunciones disponibles:")
print("  - fit_base_scaler(Z_base)")
print("  - transform_latents_with_base_scaler(Z_base, Z_T245, Z_T370, Z_T585, scaler=None)")
print("  - save_base_scaler(scaler, filepath)")
print("  - load_base_scaler(filepath)")

Utilidades de escalado consistente cargadas

Funciones disponibles:
  - fit_base_scaler(Z_base)
  - transform_latents_with_base_scaler(Z_base, Z_T245, Z_T370, Z_T585, scaler=None)
  - save_base_scaler(scaler, filepath)
  - load_base_scaler(filepath)


In [31]:
# Cargar espacios latentes existentes (ejemplo con AE)
Z_base = LATENTS["AE"]["base"]
Z_T245 = LATENTS["AE"]["T245"]
Z_T370 = LATENTS["AE"]["T370"]
Z_T585 = LATENTS["AE"]["T585"]

print("Espacios latentes cargados:")
print(f"  BASE:   {Z_base.shape}")
print(f"  SSP245: {Z_T245.shape}")
print(f"  SSP370: {Z_T370.shape}")
print(f"  SSP585: {Z_T585.shape}")

# Aplicar escalado consistente
scaled_results = transform_latents_with_base_scaler(
    Z_base=Z_base,
    Z_T245=Z_T245,
    Z_T370=Z_T370,
    Z_T585=Z_T585
)

# Extraer resultados
Z_base_sc = scaled_results['Z_base_sc']
Z_T245_sc = scaled_results['Z_T245_sc']
Z_T370_sc = scaled_results['Z_T370_sc']
Z_T585_sc = scaled_results['Z_T585_sc']
base_scaler = scaled_results['scaler']

# Guardar escalador para uso futuro
scaler_path = os.path.join(CLUSTERING_DIR, "base_scaler_AE.pkl")
save_base_scaler(base_scaler, scaler_path)

print("\nEspacios latentes escalados disponibles:")
print("  - Z_base_sc")
print("  - Z_T245_sc")
print("  - Z_T370_sc")
print("  - Z_T585_sc")
print("  - base_scaler")

Espacios latentes cargados:
  BASE:   (1983, 8)
  SSP245: (661, 8)
  SSP370: (661, 8)
  SSP585: (661, 8)
Escalador ajustado con BASE
  Shape: (1983, 8)
  Media ajustada: [-0.70330967  0.42800138  0.26757134  0.07490441  0.19109943]
  Std ajustada: [1.3451052  1.5630926  1.40206054 1.39762206 1.39828872]

Transformación completada:
  BASE:   (1983, 8) → (1983, 8)
  SSP245: (661, 8) → (661, 8)
  SSP370: (661, 8) → (661, 8)
  SSP585: (661, 8) → (661, 8)

Verificación BASE escalado:
  Media ≈ 0: False (max abs = 2.95e-07)
  Std ≈ 1:   True (max diff = 5.96e-07)

Estadísticas escenarios futuros (NO deben ser 0/1):
  SSP245 - Media: [-0.35328916 -0.21621472  0.8085322  -0.4295076  -1.3477892 ]
  SSP245 - Std:   [0.54613835 1.5512967  1.1425194  0.7996502  0.9367821 ]
  SSP370 - Media: [ 1.0290517  -0.63318604 -0.7750661  -0.03493163  0.4093097 ]
  SSP370 - Std:   [0.34723166 0.5851538  1.4075351  0.6677895  0.87963384]
  SSP585 - Media: [-1.0649142   0.8810821   0.18346253  0.9135463   1.418

In [32]:
# ========== DIAGNÓSTICO: ¿BASE está apilado? ==========
print("Análisis de estructura de datos:")
print(f"  X_BASE:     {X_BASE.shape}")
print(f"  X245_orig:  {X245_orig.shape}")
print(f"  X370_orig:  {X370_orig.shape}")
print(f"  X585_orig:  {X585_orig.shape}")
print(f"  N_PER_SCENARIO: {N_PER_SCENARIO}")

# Extraer labels_base del modelo AE
model_key = "AE"
labels_base = CLUSTERING_RESULTS[model_key]['labels_base']
centroids = CLUSTERING_RESULTS[model_key]['centroids']
print(f"\nLabels BASE cargados desde modelo {model_key}: {labels_base.shape}")

# Verificar si BASE está apilado (3 × N_PER_SCENARIO)
if X_BASE.shape[0] == 3 * N_PER_SCENARIO:
    print(f"\n✓ BASE está APILADO: {X_BASE.shape[0]} = 3 × {N_PER_SCENARIO}")
    print("  Necesitamos desapilar en tres bloques")
    
    # Desapilar BASE en tres bloques iguales
    n = N_PER_SCENARIO
    X_BASE_hist = X_BASE[:n]      # Primer tercio (histórico?)
    X_BASE_245  = X_BASE[n:2*n]   # Segundo tercio (SSP245?)
    X_BASE_370  = X_BASE[2*n:]    # Tercer tercio (SSP370? o 585?)
    
    print(f"\n  Bloques desapilados:")
    print(f"    X_BASE_hist: {X_BASE_hist.shape}")
    print(f"    X_BASE_245:  {X_BASE_245.shape}")
    print(f"    X_BASE_370:  {X_BASE_370.shape}")
    
    # Verificar dimensionalidad de features
    print(f"\n  Dimensiones de features:")
    print(f"    BASE (apilado): {X_BASE.shape[1]} features")
    print(f"    SSPs:           {X245_orig.shape[1]} features")
    print(f"    feature_names:  {len(feature_names)} nombres")
    
    # Identificar qué features faltan/sobran
    n_base_feat = X_BASE.shape[1]
    n_ssp_feat = X245_orig.shape[1]
    diff = n_ssp_feat - n_base_feat
    
    if diff > 0:
        print(f"\n  SSPs tienen {diff} features ADICIONALES")
        print(f"  Features adicionales: {feature_names[n_base_feat:n_ssp_feat]}")
        
        # Estrategia: usar solo features comunes
        print(f"\n  ESTRATEGIA: Usar primeras {n_base_feat} features comunes")
        feature_names_common = feature_names[:n_base_feat]
        
        # Recortar SSPs a features comunes
        X245_common = X245_orig[:, :n_base_feat]
        X370_common = X370_orig[:, :n_base_feat]
        X585_common = X585_orig[:, :n_base_feat]
        
        print(f"\n  SSPs recortados:")
        print(f"    X245_common: {X245_common.shape}")
        print(f"    X370_common: {X370_common.shape}")
        print(f"    X585_common: {X585_common.shape}")
        
    elif diff < 0:
        print(f"\n  ERROR: BASE tiene más features ({n_base_feat}) que SSPs ({n_ssp_feat})")
        raise ValueError("Inconsistencia dimensional - revisar origen de datos")
    else:
        print(f"\n  ✓ Mismo número de features: {n_base_feat}")
        feature_names_common = feature_names[:n_base_feat]
        X245_common = X245_orig
        X370_common = X370_orig
        X585_common = X585_orig
    
    # Desapilar labels también
    labels_base_hist = labels_base[:n]
    labels_base_245  = labels_base[n:2*n]
    labels_base_370  = labels_base[2*n:]
    
    print(f"\n  Labels desapilados:")
    print(f"    labels_base_hist: {labels_base_hist.shape}")
    print(f"    labels_base_245:  {labels_base_245.shape}")
    print(f"    labels_base_370:  {labels_base_370.shape}")
    
else:
    print(f"\n⚠ BASE NO está apilado correctamente")
    print(f"  Esperado: {3 * N_PER_SCENARIO}")
    print(f"  Real:     {X_BASE.shape[0]}")
    print("\n  Revisar cómo se construyó X_BASE en el clustering")

print("\n" + "="*70)
print("Verificación disponible para siguiente paso")
print("="*70)


Análisis de estructura de datos:
  X_BASE:     (1983, 29)
  X245_orig:  (661, 47)
  X370_orig:  (661, 47)
  X585_orig:  (661, 47)
  N_PER_SCENARIO: 661

Labels BASE cargados desde modelo AE: (1983,)

✓ BASE está APILADO: 1983 = 3 × 661
  Necesitamos desapilar en tres bloques

  Bloques desapilados:
    X_BASE_hist: (661, 29)
    X_BASE_245:  (661, 29)
    X_BASE_370:  (661, 29)

  Dimensiones de features:
    BASE (apilado): 29 features
    SSPs:           47 features
    feature_names:  47 nombres

  SSPs tienen 18 features ADICIONALES
  Features adicionales: ['climate_rx5day_decadal_mean_2020', 'climate_rx5day_decadal_mean_2050', 'climate_rx5day_decadal_mean_2080', 'climate_cdd_decadal_mean_2020', 'climate_cdd_decadal_mean_2050', 'climate_cdd_decadal_mean_2080', 'climate_sdii_decadal_mean_2020', 'climate_sdii_decadal_mean_2050', 'climate_sdii_decadal_mean_2080', 'calliope_h2_prod_ton_std_T', 'climate_cdd_std_T', 'climate_prcptot_std_T', 'climate_r10mm_std_T', 'climate_rx1day_std_T', 

In [33]:
# ========== ASIGNACIÓN DE LABELS PARA ESCENARIOS TARGET ==========
# Asignar labels a SSPs usando nearest centroid en espacio latente escalado

def assign_by_nearest_centroid(Z_sc: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """Asigna labels usando distancia euclidiana a centroides BASE."""
    # dist^2 = ||x||^2 + ||c||^2 - 2 x·c  (vectorizado)
    x2 = np.sum(Z_sc**2, axis=1, keepdims=True)              # (n,1)
    c2 = np.sum(centroids**2, axis=1)[None, :]               # (1,k)
    xc = Z_sc @ centroids.T                                  # (n,k)
    d2 = x2 + c2 - 2 * xc
    return np.argmin(d2, axis=1).astype(int)

print("Asignando labels a escenarios TARGET usando centroides BASE...")
print(f"  Centroides BASE: {centroids.shape}")
print(f"  Z_T245_sc: {Z_T245_sc.shape}")
print(f"  Z_T370_sc: {Z_T370_sc.shape}")
print(f"  Z_T585_sc: {Z_T585_sc.shape}")

labels_T245 = assign_by_nearest_centroid(Z_T245_sc, centroids)
labels_T370 = assign_by_nearest_centroid(Z_T370_sc, centroids)
labels_T585 = assign_by_nearest_centroid(Z_T585_sc, centroids)

print("\nLabels TARGET asignados:")
print(f"  labels_T245: {labels_T245.shape} - {np.unique(labels_T245).size} clústeres únicos")
print(f"  labels_T370: {labels_T370.shape} - {np.unique(labels_T370).size} clústeres únicos")
print(f"  labels_T585: {labels_T585.shape} - {np.unique(labels_T585).size} clústeres únicos")

# Verificar distribución de labels
from collections import Counter
print("\nDistribución de píxeles por clúster:")
print(f"  BASE histórico: {Counter(labels_base_hist)}")
print(f"  SSP245:         {Counter(labels_T245)}")
print(f"  SSP370:         {Counter(labels_T370)}")
print(f"  SSP585:         {Counter(labels_T585)}")


Asignando labels a escenarios TARGET usando centroides BASE...
  Centroides BASE: (10, 8)
  Z_T245_sc: (661, 8)
  Z_T370_sc: (661, 8)
  Z_T585_sc: (661, 8)

Labels TARGET asignados:
  labels_T245: (661,) - 5 clústeres únicos
  labels_T370: (661,) - 3 clústeres únicos
  labels_T585: (661,) - 4 clústeres únicos

Distribución de píxeles por clúster:
  BASE histórico: Counter({np.int32(5): 310, np.int32(3): 197, np.int32(0): 154})
  SSP245:         Counter({np.int64(5): 333, np.int64(3): 219, np.int64(0): 90, np.int64(7): 18, np.int64(2): 1})
  SSP370:         Counter({np.int64(4): 337, np.int64(6): 182, np.int64(1): 142})
  SSP585:         Counter({np.int64(8): 217, np.int64(9): 176, np.int64(2): 165, np.int64(7): 103})


In [34]:
# ========== CÁLCULO DE PERFILES EN ESPACIO ORIGINAL ==========

def cluster_profiles_original(X: np.ndarray, labels: np.ndarray, feature_names: list[str]):
    """Calcula perfiles promedio por clúster en espacio original."""
    if X.shape[1] != len(feature_names):
        raise ValueError(f"Mismatch: X tiene {X.shape[1]} columnas pero feature_names tiene {len(feature_names)} elementos")
    df = pd.DataFrame(X, columns=feature_names)
    df["cluster"] = labels
    prof = df.groupby("cluster")[feature_names].mean().reset_index()
    return prof

# Usar datos comunes (features compatibles entre BASE y SSPs)
print("Generando perfiles en espacio climático original...")
print(f"  Usando {len(feature_names_common)} features comunes")
print(f"  Features: {feature_names_common[:5]}...{feature_names_common[-3:]}")

# Perfiles BASE histórico (primer bloque)
profiles_base_hist_orig = cluster_profiles_original(
    X_BASE_hist, 
    labels_base_hist, 
    feature_names_common
)

# Perfiles SSPs (con features comunes)
profiles_T245_orig = cluster_profiles_original(
    X245_common, 
    labels_T245, 
    feature_names_common
)

profiles_T370_orig = cluster_profiles_original(
    X370_common, 
    labels_T370, 
    feature_names_common
)

profiles_T585_orig = cluster_profiles_original(
    X585_common, 
    labels_T585, 
    feature_names_common
)

print("\nPerfiles generados exitosamente:")
print(f"  BASE histórico: {profiles_base_hist_orig.shape}")
print(f"  SSP245:         {profiles_T245_orig.shape}")
print(f"  SSP370:         {profiles_T370_orig.shape}")
print(f"  SSP585:         {profiles_T585_orig.shape}")

# Mostrar muestra de perfiles BASE histórico
print("\nMuestra de perfiles BASE histórico (primeras 3 features por clúster):")
cols_to_show = ["cluster"] + feature_names_common[:3]
print(profiles_base_hist_orig[cols_to_show].head())

# Guardar perfiles
profiles_dir = CLUSTERING_DIR
profiles_base_hist_orig.to_csv(os.path.join(profiles_dir, "profiles_original_BASE_hist.csv"), index=False)
profiles_T245_orig.to_csv(os.path.join(profiles_dir, "profiles_original_T245.csv"), index=False)
profiles_T370_orig.to_csv(os.path.join(profiles_dir, "profiles_original_T370.csv"), index=False)
profiles_T585_orig.to_csv(os.path.join(profiles_dir, "profiles_original_T585.csv"), index=False)

print(f"\nPerfiles guardados en: {profiles_dir}")
print("  - profiles_original_BASE_hist.csv")
print("  - profiles_original_T245.csv")
print("  - profiles_original_T370.csv")
print("  - profiles_original_T585.csv")

Generando perfiles en espacio climático original...
  Usando 29 features comunes
  Features: ['calliope_cf_mean', 'calliope_cap_electrolyzer_mw', 'topo_slope', 'topo_elevation', 'landuse_suitable_pv']...['climate_rx1day_decadal_mean_2020', 'climate_rx1day_decadal_mean_2050', 'climate_rx1day_decadal_mean_2080']

Perfiles generados exitosamente:
  BASE histórico: (3, 30)
  SSP245:         (5, 30)
  SSP370:         (3, 30)
  SSP585:         (4, 30)

Muestra de perfiles BASE histórico (primeras 3 features por clúster):
   cluster  calliope_cf_mean  calliope_cap_electrolyzer_mw  topo_slope
0        0         -0.555990                     -0.730494    1.138192
1        3          0.663143                      0.055552   -1.042277
2        5         -0.437736                     -0.890899    0.203670

Perfiles guardados en: /home/aninotna/magister/tesis/justh2_pipeline/data/autoencoder_results/clustering
  - profiles_original_BASE_hist.csv
  - profiles_original_T245.csv
  - profiles_original_

## Etiquetado Climático y Energético de Clústeres

Sistema basado en **percentiles sobre datos RAW** (píxel a píxel) para asignar etiquetas interpretables:

**CLIMA:**
- **Temperatura**: cálido, frío (comparación de medianas clúster vs global)
- **Precipitación**: húmedo, seco (ratio p50 clúster/global)
- **Sequía**: CDD alto (días consecutivos secos)
- **Extremos**: RX1day, RX5day, R95p, R99p (percentil 75 clúster vs global)
- **Variabilidad**: alta desviación estándar

**ENERGÍA:**
- **Producción H₂**: alta/baja producción (±20% vs mediana)
- **Capacity Factor**: alto/bajo CF (±15% vs mediana, eficiencia de generación renovable)

**Método**: Calcular percentiles (25, 50, 75) por clúster en espacio original → comparar con percentiles globales POR ESCENARIO → asignar etiquetas según ratios.

In [38]:
# ========== DIAGNÓSTICO: Verificar detección de variables ==========
print("DIAGNÓSTICO: Variables detectadas por cada patrón")
print("="*70)

for cat, cols in COLS.items():
    print(f"\n{cat:8s}: {len(cols):2d} variables")
    if cols:
        for c in cols:
            print(f"  - {c}")
    else:
        print(f"  ⚠️  NO SE DETECTARON VARIABLES (esto causará RuntimeWarning)")

print("\n" + "="*70)
print("Variables PROBLEMÁTICAS esperadas:")
print("  - CDD (días consecutivos secos)")
print("  - RX5day (precipitación máxima en 5 días)")
print("  - R95p, R99p (percentiles de precipitación)")
print("  - SPI (índice de precipitación estandarizado)")
print("  - STD/CV (variabilidad)")
print("="*70)

DIAGNÓSTICO: Variables detectadas por cada patrón

tmax    :  3 variables
  - climate_tmax_mean_decadal_mean_2020
  - climate_tmax_mean_decadal_mean_2050
  - climate_tmax_mean_decadal_mean_2080

tmin    :  3 variables
  - climate_tmin_mean_decadal_mean_2020
  - climate_tmin_mean_decadal_mean_2050
  - climate_tmin_mean_decadal_mean_2080

pr      :  6 variables
  - climate_prcptot_decadal_mean_2020
  - climate_prcptot_decadal_mean_2050
  - climate_prcptot_decadal_mean_2080
  - climate_r10mm_decadal_mean_2020
  - climate_r10mm_decadal_mean_2050
  - climate_r10mm_decadal_mean_2080

rx1     :  3 variables
  - climate_rx1day_decadal_mean_2020
  - climate_rx1day_decadal_mean_2050
  - climate_rx1day_decadal_mean_2080

rx5     :  0 variables
  ⚠️  NO SE DETECTARON VARIABLES (esto causará RuntimeWarning)

r95     :  0 variables
  ⚠️  NO SE DETECTARON VARIABLES (esto causará RuntimeWarning)

r99     :  0 variables
  ⚠️  NO SE DETECTARON VARIABLES (esto causará RuntimeWarning)

sdii    :  0 variab

In [40]:
# ================================================
# ETIQUETADO CLIMÁTICO CON PERCENTILES POR ESCENARIO
# ================================================
import re

# --- 1) Definir patrones de búsqueda para variables climáticas Y ENERGÉTICAS ---
# Patrones más estrictos para evitar falsos positivos
PAT = {
    # Variables climáticas
    "tmax": re.compile(r"climate.*tmax|climate.*tasmax", re.I),
    "tmin": re.compile(r"climate.*tmin|climate.*tasmin", re.I),
    "pr":   re.compile(r"climate.*prcptot|climate.*r\d+mm", re.I),  # Excluir h2_prod, conflict_proximity
    "rx1":  re.compile(r"climate.*rx1day", re.I),
    "rx5":  re.compile(r"climate.*rx5day", re.I),
    "r95":  re.compile(r"climate.*r95p", re.I),
    "r99":  re.compile(r"climate.*r99p", re.I),
    "sdii": re.compile(r"climate.*sdii", re.I),
    "cdd":  re.compile(r"climate.*cdd", re.I),
    "cwd":  re.compile(r"climate.*cwd", re.I),
    "spi":  re.compile(r"climate.*spi", re.I),
    "var":  re.compile(r"climate.*(std|_std_)", re.I),
    # Variables energéticas (NUEVAS)
    "h2_prod": re.compile(r"h2_prod_ton", re.I),  # Producción de H₂ en toneladas
    "cf": re.compile(r"calliope_cf_mean", re.I),  # Capacity factor medio
}

def find_cols(names, rx):
    """Encuentra columnas que coinciden con el patrón regex."""
    return [c for c in names if rx.search(c)]

COLS = {k: find_cols(feature_names_common, rx) for k, rx in PAT.items()}

print("Variables detectadas por categoría:")
print("\n--- CLIMÁTICAS ---")
for cat in ["tmax", "tmin", "pr", "rx1", "rx5", "r95", "r99", "sdii", "cdd", "cwd", "spi", "var"]:
    cols = COLS.get(cat, [])
    if cols:
        print(f"  {cat:8s}: {len(cols):2d} vars → {cols[:3]}{'...' if len(cols)>3 else ''}")

print("\n--- ENERGÉTICAS ---")
for cat in ["h2_prod", "cf"]:
    cols = COLS.get(cat, [])
    if cols:
        print(f"  {cat:8s}: {len(cols):2d} vars → {cols[:3]}{'...' if len(cols)>3 else ''}")

# --- 2) Calcular percentiles por clúster en datos RAW (píxel a píxel) ---
def percentile_stats_per_cluster(X_raw, labels, feat_names, percentiles=[25, 50, 75]):
    """
    Calcula percentiles de cada variable climática por clúster.
    Retorna DataFrame con cluster_id × variables con percentiles especificados.
    """
    stats_list = []
    unique_clusters = np.unique(labels)
    
    for cid in unique_clusters:
        idx = np.where(labels == cid)[0]
        X_cluster = X_raw[idx]  # píxeles del clúster
        
        row = {"cluster": int(cid), "n_pixels": len(idx)}
        
        for var_name in feat_names:
            col_idx = feat_names.index(var_name)
            vals = X_cluster[:, col_idx]
            
            # Calcular percentiles
            for p in percentiles:
                row[f"{var_name}_p{p}"] = np.percentile(vals, p)
        
        stats_list.append(row)
    
    return pd.DataFrame(stats_list)

print("\nCalculando percentiles en datos RAW por clúster...")
print("  Percentiles: 25th (Q1), 50th (mediana), 75th (Q3)")

perc_base_hist = percentile_stats_per_cluster(X_BASE_hist, labels_base_hist, feature_names_common)
perc_245 = percentile_stats_per_cluster(X245_common, labels_T245, feature_names_common)
perc_370 = percentile_stats_per_cluster(X370_common, labels_T370, feature_names_common)
perc_585 = percentile_stats_per_cluster(X585_common, labels_T585, feature_names_common)

print(f"\n  BASE histórico: {perc_base_hist.shape}")
print(f"  SSP245:         {perc_245.shape}")
print(f"  SSP370:         {perc_370.shape}")
print(f"  SSP585:         {perc_585.shape}")

# --- 3) Calcular percentiles GLOBALES POR ESCENARIO (Opción A) ---
def global_percentiles(X_raw, feat_names, percentiles=[25, 50, 75]):
    """Calcula percentiles globales sobre todos los píxeles de UN escenario."""
    global_stats = {}
    for var_name in feat_names:
        col_idx = feat_names.index(var_name)
        vals = X_raw[:, col_idx]
        for p in percentiles:
            global_stats[f"{var_name}_p{p}"] = np.percentile(vals, p)
    return global_stats

print("\n" + "="*70)
print("OPCIÓN A: Percentiles globales POR ESCENARIO INDIVIDUAL")
print("="*70)
print("Cada escenario usa sus propios percentiles como referencia.")
print("Esto permite detectar diversidad climática DENTRO de cada futuro.")

global_p_base = global_percentiles(X_BASE_hist, feature_names_common)
global_p_245 = global_percentiles(X245_common, feature_names_common)
global_p_370 = global_percentiles(X370_common, feature_names_common)
global_p_585 = global_percentiles(X585_common, feature_names_common)

print(f"\n✓ Percentiles globales calculados para cada escenario:")
print(f"  BASE histórico: {len(feature_names_common)} variables")
print(f"  SSP245:         {len(feature_names_common)} variables")
print(f"  SSP370:         {len(feature_names_common)} variables")
print(f"  SSP585:         {len(feature_names_common)} variables")

# --- 4) Funciones de etiquetado basadas en percentiles ---
def mean_percentile_over(row, cols, suffix="_p50"):
    """Promedio de medianas (p50) sobre un conjunto de columnas."""
    if not cols:
        return np.nan
    col_names = [f"{c}{suffix}" for c in cols]
    existing = [c for c in col_names if c in row.index]
    if not existing:
        return np.nan
    vals = row[existing].values.astype(float)
    return float(np.nanmean(vals)) if len(vals) else np.nan

def relative_position(cluster_val, global_val):
    """
    Calcula posición relativa del clúster vs global.
    Retorna ratio: cluster/global (si global != 0)
    """
    if global_val == 0:
        return 1.0
    return cluster_val / global_val

def label_cluster_percentile(row, feat_names, global_stats, thresholds=None):
    """
    Asigna etiquetas climáticas Y ENERGÉTICAS a un clúster comparando sus percentiles
    con los percentiles globales DEL MISMO ESCENARIO.
    
    Umbrales balanceados para capturar diversidad interna:
    
    CLIMA:
    - Cálido: > 1.15x mediana global (15% más cálido)
    - Frío:   < 0.85x mediana global (15% más frío)
    - Húmedo: > 1.2x mediana global (20% más lluvia)
    - Seco:   < 0.8x mediana global (20% menos lluvia)
    - Extremos: > 1.3x p75 global (30% más extremo)
    
    ENERGÍA:
    - Alta producción H₂: > 1.2x mediana global (20% más producción)
    - Baja producción H₂: < 0.8x mediana global (20% menos producción)
    - Alto CF: > 1.15x mediana global (15% más eficiente)
    - Bajo CF: < 0.85x mediana global (15% menos eficiente)
    """
    if thresholds is None:
        thresholds = {
            # Umbrales climáticos
            "temp_hot": 1.15,     # 15% más cálido que la mediana
            "temp_cold": 0.85,    # 15% más frío
            "pr_wet": 1.2,        # 20% más húmedo
            "pr_dry": 0.8,        # 20% más seco
            "extreme": 1.3,       # 30% más extremos
            "drought": 1.25,      # 25% más sequía
            # Umbrales energéticos (NUEVOS)
            "h2_high": 1.2,       # 20% más producción H₂
            "h2_low": 0.8,        # 20% menos producción H₂
            "cf_high": 1.15,      # 15% más eficiencia CF
            "cf_low": 0.85,       # 15% menos eficiencia CF
        }
    
    tags = []

    # Temperatura (usar mediana - p50)
    tmax_cluster = mean_percentile_over(row, COLS["tmax"], "_p50")
    tmin_cluster = mean_percentile_over(row, COLS["tmin"], "_p50")
    
    tmax_global = np.nanmean([global_stats.get(f"{c}_p50", np.nan) for c in COLS["tmax"]])
    tmin_global = np.nanmean([global_stats.get(f"{c}_p50", np.nan) for c in COLS["tmin"]])
    
    if not np.isnan(tmax_cluster) and not np.isnan(tmax_global) and tmax_global != 0:
        ratio_tmax = relative_position(tmax_cluster, tmax_global)
        if ratio_tmax > thresholds["temp_hot"]:
            tags.append("cálido")
        elif ratio_tmax < thresholds["temp_cold"]:
            tags.append("frío")
    
    if not np.isnan(tmin_cluster) and not np.isnan(tmin_global) and tmin_global != 0:
        ratio_tmin = relative_position(tmin_cluster, tmin_global)
        if ratio_tmin > thresholds["temp_hot"] and "cálido" not in tags:
            tags.append("cálido")
        elif ratio_tmin < thresholds["temp_cold"] and "frío" not in tags:
            tags.append("frío")

    # Precipitación
    pr_cluster = mean_percentile_over(row, COLS["pr"], "_p50")
    pr_global = np.nanmean([global_stats.get(f"{c}_p50", np.nan) for c in COLS["pr"]])
    
    if not np.isnan(pr_cluster) and not np.isnan(pr_global) and pr_global > 0:
        ratio_pr = relative_position(pr_cluster, pr_global)
        if ratio_pr > thresholds["pr_wet"]:
            tags.append("húmedo")
        elif ratio_pr < thresholds["pr_dry"]:
            tags.append("seco")

    # Sequía (CDD) - SOLO si existe la variable
    if COLS["cdd"]:
        cdd_cluster = mean_percentile_over(row, COLS["cdd"], "_p50")
        cdd_global = np.nanmean([global_stats.get(f"{c}_p50", np.nan) for c in COLS["cdd"]])
        
        if not np.isnan(cdd_cluster) and not np.isnan(cdd_global) and cdd_global > 0:
            ratio_cdd = relative_position(cdd_cluster, cdd_global)
            if ratio_cdd > thresholds["drought"]:
                tags.append("sequía prolongada")

    # Extremos de precipitación (usar p75 para capturar eventos extremos)
    rx_vals = []
    if COLS["rx1"]:
        rx_vals.append(mean_percentile_over(row, COLS["rx1"], "_p75"))
    if COLS["rx5"]:
        rx_vals.append(mean_percentile_over(row, COLS["rx5"], "_p75"))
    
    if rx_vals:  # Solo calcular si hay al menos una variable
        rx_cluster = np.nanmean(rx_vals)
        
        rx_global_vals = []
        if COLS["rx1"]:
            rx_global_vals.append(np.nanmean([global_stats.get(f"{c}_p75", np.nan) for c in COLS["rx1"]]))
        if COLS["rx5"]:
            rx_global_vals.append(np.nanmean([global_stats.get(f"{c}_p75", np.nan) for c in COLS["rx5"]]))
        
        rx_global = np.nanmean(rx_global_vals)
        
        if not np.isnan(rx_cluster) and not np.isnan(rx_global) and rx_global > 0:
            ratio_rx = relative_position(rx_cluster, rx_global)
            if ratio_rx > thresholds["extreme"]:
                tags.append("extremos de lluvia")

    # R95p, R99p - SOLO si existen
    if COLS["r95"]:
        r95_cluster = mean_percentile_over(row, COLS["r95"], "_p75")
        r95_global = np.nanmean([global_stats.get(f"{c}_p75", np.nan) for c in COLS["r95"]])
        
        if not np.isnan(r95_cluster) and not np.isnan(r95_global) and r95_global > 0:
            ratio_r95 = relative_position(r95_cluster, r95_global)
            if ratio_r95 > thresholds["extreme"] and "extremos de lluvia" not in tags:
                tags.append("extremos de lluvia")

    # SPI (anomalía seca) - SOLO si existe
    if COLS["spi"]:
        spi_cluster = mean_percentile_over(row, COLS["spi"], "_p50")
        spi_global = np.nanmean([global_stats.get(f"{c}_p50", np.nan) for c in COLS["spi"]])
        
        if not np.isnan(spi_cluster) and not np.isnan(spi_global):
            if spi_cluster < spi_global - 0.5:  # SPI < global - 0.5
                tags.append("anomalía seca (SPI)")

    # Variabilidad (usar p75 de std/cv) - SOLO si existe
    if COLS["var"]:
        var_cluster = mean_percentile_over(row, COLS["var"], "_p75")
        var_global = np.nanmean([global_stats.get(f"{c}_p75", np.nan) for c in COLS["var"]])
        
        if not np.isnan(var_cluster) and not np.isnan(var_global) and var_global > 0:
            ratio_var = relative_position(var_cluster, var_global)
            if ratio_var > 1.2:
                tags.append("alta variabilidad")

    # ========== VARIABLES ENERGÉTICAS (H₂ y CF) ==========
    
    # Producción de H₂ (usar mediana p50)
    if COLS["h2_prod"]:
        h2_cluster = mean_percentile_over(row, COLS["h2_prod"], "_p50")
        h2_global = np.nanmean([global_stats.get(f"{c}_p50", np.nan) for c in COLS["h2_prod"]])
        
        if not np.isnan(h2_cluster) and not np.isnan(h2_global) and h2_global > 0:
            ratio_h2 = relative_position(h2_cluster, h2_global)
            if ratio_h2 > thresholds["h2_high"]:
                tags.append("alta producción H₂")
            elif ratio_h2 < thresholds["h2_low"]:
                tags.append("baja producción H₂")
    
    # Capacity Factor (eficiencia de generación renovable)
    if COLS["cf"]:
        cf_cluster = mean_percentile_over(row, COLS["cf"], "_p50")
        cf_global = np.nanmean([global_stats.get(f"{c}_p50", np.nan) for c in COLS["cf"]])
        
        if not np.isnan(cf_cluster) and not np.isnan(cf_global) and cf_global > 0:
            ratio_cf = relative_position(cf_cluster, cf_global)
            if ratio_cf > thresholds["cf_high"]:
                tags.append("alto CF")
            elif ratio_cf < thresholds["cf_low"]:
                tags.append("bajo CF")

    # Etiquetas compuestas
    if "cálido" in tags and "seco" in tags:
        tags.append("cálido-seco")
    if "cálido" in tags and "húmedo" in tags:
        tags.append("cálido-húmedo")
    if "frío" in tags and "húmedo" in tags:
        tags.append("frío-húmedo")
    if "frío" in tags and "seco" in tags:
        tags.append("frío-seco")

    # Remover duplicados conservando orden
    seen = set()
    clean = []
    for t in tags:
        if t not in seen:
            seen.add(t)
            clean.append(t)
    
    return ", ".join(clean) if clean else "neutro/mixto"

def label_table_percentile(perc_df, feat_names, global_stats, tag):
    """Genera tabla de etiquetas climáticas para todos los clústeres."""
    out = []
    for _, r in perc_df.iterrows():
        cid = int(r["cluster"])
        lab = label_cluster_percentile(r, feat_names, global_stats)
        out.append({"scenario": tag, "cluster": cid, "label": lab})
    return pd.DataFrame(out)

# --- 5) Generar etiquetas usando percentiles del MISMO escenario ---
print("\n" + "="*70)
print("Generando etiquetas climáticas Y ENERGÉTICAS (percentiles por escenario)...")
print("Umbrales CLIMA: Cálido/Frío ±15%, Húmedo/Seco ±20%, Extremos +30%")
print("Umbrales ENERGÍA: H₂ ±20%, CF ±15%")
print("="*70)

labels_base_hist_tbl = label_table_percentile(perc_base_hist, feature_names_common, global_p_base, "BASE_hist")
labels_245_tbl = label_table_percentile(perc_245, feature_names_common, global_p_245, "T245")
labels_370_tbl = label_table_percentile(perc_370, feature_names_common, global_p_370, "T370")
labels_585_tbl = label_table_percentile(perc_585, feature_names_common, global_p_585, "T585")

# --- 6) Mostrar resultados ---
print("\n" + "="*70)
print("ETIQUETAS CLIMÁTICAS Y ENERGÉTICAS POR CLÚSTER")
print("(Basadas en percentiles del MISMO escenario)")
print("="*70)

print("\n=== BASE histórico ===")
for _, r in labels_base_hist_tbl.sort_values("cluster").iterrows():
    print(f"  Clúster {r['cluster']:2d}: {r['label']}")

print("\n=== SSP245 ===")
for _, r in labels_245_tbl.sort_values("cluster").iterrows():
    print(f"  Clúster {r['cluster']:2d}: {r['label']}")

print("\n=== SSP370 ===")
for _, r in labels_370_tbl.sort_values("cluster").iterrows():
    print(f"  Clúster {r['cluster']:2d}: {r['label']}")

print("\n=== SSP585 ===")
for _, r in labels_585_tbl.sort_values("cluster").iterrows():
    print(f"  Clúster {r['cluster']:2d}: {r['label']}")

# --- 7) Guardar a disco ---
labels_base_hist_tbl.to_csv(os.path.join(CLUSTERING_DIR, "labels_climate_AE_BASE_hist.csv"), index=False)
labels_245_tbl.to_csv(os.path.join(CLUSTERING_DIR, "labels_climate_AE_T245.csv"), index=False)
labels_370_tbl.to_csv(os.path.join(CLUSTERING_DIR, "labels_climate_AE_T370.csv"), index=False)
labels_585_tbl.to_csv(os.path.join(CLUSTERING_DIR, "labels_climate_AE_T585.csv"), index=False)

# Guardar también los percentiles
perc_base_hist.to_csv(os.path.join(CLUSTERING_DIR, "percentiles_BASE_hist.csv"), index=False)
perc_245.to_csv(os.path.join(CLUSTERING_DIR, "percentiles_T245.csv"), index=False)
perc_370.to_csv(os.path.join(CLUSTERING_DIR, "percentiles_T370.csv"), index=False)
perc_585.to_csv(os.path.join(CLUSTERING_DIR, "percentiles_T585.csv"), index=False)

# Guardar percentiles globales para referencia
import json
global_stats = {
    "BASE_hist": {k: float(v) if not np.isnan(v) else None for k, v in global_p_base.items()},
    "T245": {k: float(v) if not np.isnan(v) else None for k, v in global_p_245.items()},
    "T370": {k: float(v) if not np.isnan(v) else None for k, v in global_p_370.items()},
    "T585": {k: float(v) if not np.isnan(v) else None for k, v in global_p_585.items()},
}
with open(os.path.join(CLUSTERING_DIR, "global_percentiles_by_scenario.json"), "w") as f:
    json.dump(global_stats, f, indent=2)

print("\n" + "="*70)
print(f"Resultados guardados en: {CLUSTERING_DIR}")
print("  Etiquetas:")
print("    - labels_climate_AE_BASE_hist.csv")
print("    - labels_climate_AE_T245.csv")
print("    - labels_climate_AE_T370.csv")
print("    - labels_climate_AE_T585.csv")
print("  Percentiles por clúster:")
print("    - percentiles_BASE_hist.csv")
print("    - percentiles_T245.csv")
print("    - percentiles_T370.csv")
print("    - percentiles_T585.csv")
print("  Percentiles globales:")
print("    - global_percentiles_by_scenario.json")
print("="*70)


Variables detectadas por categoría:

--- CLIMÁTICAS ---
  tmax    :  3 vars → ['climate_tmax_mean_decadal_mean_2020', 'climate_tmax_mean_decadal_mean_2050', 'climate_tmax_mean_decadal_mean_2080']
  tmin    :  3 vars → ['climate_tmin_mean_decadal_mean_2020', 'climate_tmin_mean_decadal_mean_2050', 'climate_tmin_mean_decadal_mean_2080']
  pr      :  6 vars → ['climate_prcptot_decadal_mean_2020', 'climate_prcptot_decadal_mean_2050', 'climate_prcptot_decadal_mean_2080']...
  rx1     :  3 vars → ['climate_rx1day_decadal_mean_2020', 'climate_rx1day_decadal_mean_2050', 'climate_rx1day_decadal_mean_2080']

--- ENERGÉTICAS ---
  h2_prod :  3 vars → ['calliope_h2_prod_ton_decadal_mean_2020', 'calliope_h2_prod_ton_decadal_mean_2050', 'calliope_h2_prod_ton_decadal_mean_2080']
  cf      :  1 vars → ['calliope_cf_mean']

Calculando percentiles en datos RAW por clúster...
  Percentiles: 25th (Q1), 50th (mediana), 75th (Q3)

  BASE histórico: (3, 89)
  SSP245:         (5, 89)
  SSP370:         (3, 89)


In [42]:
# ========== DIAGNÓSTICO: Variabilidad de H₂ por escenario ==========
print("DIAGNÓSTICO: Producción de H₂ por clúster y escenario")
print("="*70)

def analyze_h2_production(perc_df, global_stats, scenario_name):
    """Analiza la producción de H₂ en cada clúster."""
    h2_vars = COLS["h2_prod"]
    if not h2_vars:
        print(f"\n{scenario_name}: NO HAY VARIABLES DE H₂")
        return
    
    print(f"\n{scenario_name}:")
    print(f"  Variables H₂: {h2_vars}")
    
    # Calcular mediana global
    h2_global = np.nanmean([global_stats.get(f"{c}_p50", np.nan) for c in h2_vars])
    print(f"  Mediana GLOBAL: {h2_global:.2f} ton/año")
    
    # Analizar cada clúster
    print(f"\n  Producción por clúster:")
    for _, row in perc_df.iterrows():
        cid = int(row["cluster"])
        h2_cluster = mean_percentile_over(row, h2_vars, "_p50")
        
        if not np.isnan(h2_cluster) and not np.isnan(h2_global) and h2_global > 0:
            ratio = h2_cluster / h2_global
            deviation = (ratio - 1.0) * 100  # Desviación porcentual
            
            # Determinar etiqueta
            if ratio > 1.2:
                label = "ALTA (+20%)"
            elif ratio < 0.8:
                label = "BAJA (-20%)"
            else:
                label = f"neutro ({deviation:+.1f}%)"
            
            print(f"    C{cid:02d}: {h2_cluster:6.2f} ton/año  (ratio={ratio:.3f}, {label})")
        else:
            print(f"    C{cid:02d}: datos insuficientes")
    
    # Calcular coeficiente de variación
    h2_values = []
    for _, row in perc_df.iterrows():
        h2_cluster = mean_percentile_over(row, h2_vars, "_p50")
        if not np.isnan(h2_cluster):
            h2_values.append(h2_cluster)
    
    if h2_values:
        cv = np.std(h2_values) / np.mean(h2_values) * 100
        print(f"\n  Coeficiente de variación: {cv:.1f}%")
        if cv < 10:
            print(f"  ⚠️  BAJA VARIABILIDAD: Producción muy uniforme entre clústeres")
        elif cv < 20:
            print(f"  ⚙️  VARIABILIDAD MODERADA: Diferencias sutiles entre clústeres")
        else:
            print(f"  ✓ ALTA VARIABILIDAD: Diferencias claras entre clústeres")

print("\nAnalizando producción de H₂ en cada escenario...")
analyze_h2_production(perc_base_hist, global_p_base, "BASE histórico")
analyze_h2_production(perc_245, global_p_245, "SSP245")
analyze_h2_production(perc_370, global_p_370, "SSP370")
analyze_h2_production(perc_585, global_p_585, "SSP585")

print("\n" + "="*70)
print("INTERPRETACIÓN:")
print("  - Si CV < 10%: homogeneización espacial (todos producen similar)")
print("  - Si ratio ∈ [0.8, 1.2]: dentro del rango 'neutro' (±20%)")
print("  - Solución: reducir umbrales (ej: ±10% → ratio ∈ [0.9, 1.1])")
print("="*70)

DIAGNÓSTICO: Producción de H₂ por clúster y escenario

Analizando producción de H₂ en cada escenario...

BASE histórico:
  Variables H₂: ['calliope_h2_prod_ton_decadal_mean_2020', 'calliope_h2_prod_ton_decadal_mean_2050', 'calliope_h2_prod_ton_decadal_mean_2080']
  Mediana GLOBAL: 0.15 ton/año

  Producción por clúster:
    C00:  -0.67 ton/año  (ratio=-4.616, BAJA (-20%))
    C03:   1.08 ton/año  (ratio=7.412, ALTA (+20%))
    C05:   0.10 ton/año  (ratio=0.671, BAJA (-20%))

  Coeficiente de variación: 426.0%
  ✓ ALTA VARIABILIDAD: Diferencias claras entre clústeres

SSP245:
  Variables H₂: ['calliope_h2_prod_ton_decadal_mean_2020', 'calliope_h2_prod_ton_decadal_mean_2050', 'calliope_h2_prod_ton_decadal_mean_2080']
  Mediana GLOBAL: 2390.20 ton/año

  Producción por clúster:
    C00: 2234.76 ton/año  (ratio=0.935, neutro (-6.5%))
    C02: 2337.62 ton/año  (ratio=0.978, neutro (-2.2%))
    C03: 2397.78 ton/año  (ratio=1.003, neutro (+0.3%))
    C05: 2460.45 ton/año  (ratio=1.029, neutro

In [41]:
# ================================================
# ANÁLISIS DE DIVERSIDAD DE ETIQUETAS
# ================================================

print("ANÁLISIS DE DIVERSIDAD EN ETIQUETADO CLIMÁTICO")
print("="*70)

# 1) Contar etiquetas únicas por escenario
def unique_label_analysis(label_df, scenario_name):
    """Analiza diversidad de etiquetas en un escenario."""
    labels = label_df["label"].values
    unique = label_df["label"].nunique()
    total = len(label_df)
    
    print(f"\n{scenario_name}:")
    print(f"  Total de clústeres: {total}")
    print(f"  Etiquetas únicas:   {unique}")
    print(f"  Diversidad:         {unique/total*100:.1f}%")
    
    # Contar frecuencia de cada etiqueta
    from collections import Counter
    counts = Counter(labels)
    print(f"  Distribución de etiquetas:")
    for label, count in counts.most_common():
        print(f"    - {label}: {count} clústeres")
    
    return unique, counts

u_base, c_base = unique_label_analysis(labels_base_hist_tbl, "BASE histórico")
u_245, c_245 = unique_label_analysis(labels_245_tbl, "SSP245")
u_370, c_370 = unique_label_analysis(labels_370_tbl, "SSP370")
u_585, c_585 = unique_label_analysis(labels_585_tbl, "SSP585")

# 2) Comparar cambios BASE → SSPs
print("\n" + "="*70)
print("CAMBIOS DE ETIQUETAS: BASE → SSPs")
print("="*70)

def compare_labels(base_df, target_df, scenario_name):
    """Compara etiquetas entre BASE y un escenario futuro."""
    print(f"\n{scenario_name}:")
    changes = 0
    maintains = 0
    
    # Obtener clústeres presentes en BASE
    base_clusters = set(base_df["cluster"].values)
    target_clusters = set(target_df["cluster"].values)
    
    # Solo comparar clústeres que existen en ambos
    common_clusters = base_clusters.intersection(target_clusters)
    
    for cid in sorted(common_clusters):
        base_label = base_df[base_df["cluster"] == cid]["label"].values[0]
        target_label = target_df[target_df["cluster"] == cid]["label"].values[0]
        
        if base_label != target_label:
            changes += 1
            print(f"  C{cid:02d}: {base_label} → {target_label}")
        else:
            maintains += 1
    
    # Reportar clústeres nuevos/desaparecidos
    new_clusters = target_clusters - base_clusters
    lost_clusters = base_clusters - target_clusters
    
    if new_clusters:
        print(f"\n  Clústeres NUEVOS en {scenario_name}: {sorted(new_clusters)}")
        for cid in sorted(new_clusters):
            label = target_df[target_df["cluster"] == cid]["label"].values[0]
            print(f"    C{cid:02d}: {label}")
    
    if lost_clusters:
        print(f"\n  Clústeres DESAPARECIDOS de BASE: {sorted(lost_clusters)}")
    
    print(f"\n  Resumen (clústeres comunes={len(common_clusters)}):")
    print(f"    Mantiene: {maintains} clústeres ({maintains/len(common_clusters)*100 if common_clusters else 0:.1f}%)")
    print(f"    Cambia:   {changes} clústeres ({changes/len(common_clusters)*100 if common_clusters else 0:.1f}%)")
    
    return changes, maintains

ch_245, ma_245 = compare_labels(labels_base_hist_tbl, labels_245_tbl, "SSP245")
ch_370, ma_370 = compare_labels(labels_base_hist_tbl, labels_370_tbl, "SSP370")
ch_585, ma_585 = compare_labels(labels_base_hist_tbl, labels_585_tbl, "SSP585")

# 3) Resumen de diversidad
print("\n" + "="*70)
print("RESUMEN DE DIVERSIDAD")
print("="*70)
print(f"\nEtiquetas únicas por escenario:")
print(f"  BASE:   {u_base} / 3 clústeres ({u_base/3*100:.0f}%)")
print(f"  SSP245: {u_245} / 5 clústeres ({u_245/5*100:.0f}%)")
print(f"  SSP370: {u_370} / 3 clústeres ({u_370/3*100:.0f}%)")
print(f"  SSP585: {u_585} / 4 clústeres ({u_585/4*100:.0f}%)")

print(f"\nCambios en clústeres comunes respecto a BASE:")
total_base = len(labels_base_hist_tbl)
print(f"  SSP245: {ch_245} clústeres cambian")
print(f"  SSP370: {ch_370} clústeres cambian")
print(f"  SSP585: {ch_585} clústeres cambian")

print("\n" + "="*70)
print("⚠️  DIAGNÓSTICO:")
if u_245 == 1 and u_370 == 1 and u_585 == 1:
    print("Todos los clústeres SSP tienen la misma etiqueta.")
    print("Posibles causas:")
    print("  1. Umbrales de percentiles muy laxos (1.15, 1.2x global)")
    print("  2. Percentiles globales calculados solo sobre BASE histórico")
    print("  3. Variables detectadas incorrectamente (pr incluye h2_prod, conflict)")
    print("\nSoluciones sugeridas:")
    print("  - Usar umbrales más estrictos (1.3x, 1.5x)")
    print("  - Calcular percentiles globales sobre BASE+SSPs combinados")
    print("  - Corregir patrones regex para variables climáticas")
print("="*70)


ANÁLISIS DE DIVERSIDAD EN ETIQUETADO CLIMÁTICO

BASE histórico:
  Total de clústeres: 3
  Etiquetas únicas:   2
  Diversidad:         66.7%
  Distribución de etiquetas:
    - cálido, baja producción H₂: 2 clústeres
    - frío, alta producción H₂: 1 clústeres

SSP245:
  Total de clústeres: 5
  Etiquetas únicas:   3
  Diversidad:         60.0%
  Distribución de etiquetas:
    - cálido, húmedo, cálido-húmedo: 2 clústeres
    - neutro/mixto: 2 clústeres
    - cálido: 1 clústeres

SSP370:
  Total de clústeres: 3
  Etiquetas únicas:   1
  Diversidad:         33.3%
  Distribución de etiquetas:
    - neutro/mixto: 3 clústeres

SSP585:
  Total de clústeres: 4
  Etiquetas únicas:   2
  Diversidad:         50.0%
  Distribución de etiquetas:
    - neutro/mixto: 3 clústeres
    - cálido: 1 clústeres

CAMBIOS DE ETIQUETAS: BASE → SSPs

SSP245:
  C00: cálido, baja producción H₂ → cálido
  C03: frío, alta producción H₂ → neutro/mixto
  C05: cálido, baja producción H₂ → neutro/mixto

  Clústeres NUEVOS